<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6C_Cell_6C_4J0_Leave_One_Gene_Out_Validation_Materialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# STAGE 6C STEP 4J — CELL 6C-4J0
# LEAVE-ONE-GENE-OUT (LOGO) VALIDATION RESULT-CATEGORY MATERIALIZATION
# ==================================================================================================
#
# Purpose
# -------
# 1. Freshly verify the immutable Stage 4B weak-label table, Stage 4C fitted-model bundles,
#    Stage 6B primary-evaluable cohort, and the completed Cell 6C-4I0 manifest.
# 2. Reproduce the exact score-blind Cell 6C-3H0 train-two/test-one design for BRCA1, BRCA2,
#    and MLH1 by cloning the checksum-verified Stage 4C pipelines and fitting only T0 weak labels.
# 3. Generate every held-out LOGO prediction before the Stage 6B temporal outcome is opened.
# 4. Reproduce the exact 2,000-attempt paired ordinary row-bootstrap stream per held-out gene,
#    including tie-aware AUPRC/AUROC and separate nine-test Holm families.
# 5. Materialize versioned accounting, prediction, replicate, interval, paired-inference,
#    historical-concordance, QC, manifest, and SHA-256 sidecar artifacts.
#
# Scientific boundary
# -------------------
# - No T1 outcome is used to fit, tune, calibrate, threshold, or select a LOGO model.
# - EGFR is not pooled into the primary LOGO experiment.
# - No frozen score, weak label, outcome, feature definition, model specification, threshold,
#   gene assignment, linkage decision, row order, or cohort membership is changed.
# - Experiment 2 is not started.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import sys
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. LOCKED INPUTS, EXPECTATIONS, AND OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_NAME = (
    "GES_Stage6C_Cell_6C_4J0_Leave_One_Gene_Out_Validation_"
    "Materialization.ipynb"
)

ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
STAGE4_DATA_DIR = ROOT / "data_processed/stage4_ges"
STAGE4_MODEL_DIR = ROOT / "models/stage4_ges"
STAGE4_CONFIG_DIR = ROOT / "configs/stage4_ges"
STAGE6_DIR = ROOT / "data_processed/stage6_temporal_validation"

WEAK_LABEL_TABLE = STAGE4_DATA_DIR / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
STAGE4B_MANIFEST = STAGE4_CONFIG_DIR / "stage4b_weak_label_freeze_manifest_v1.json"
FULL_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_full_ges_logistic_model_v1.joblib"
NO_STAR_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_no_star_ges_logistic_model_v1.joblib"
STAGE4C_MANIFEST = STAGE4_CONFIG_DIR / "stage4c_ges_model_freeze_manifest_v1.json"
EVALUABLE_PARQUET = STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
EVALUABLE_SIDECAR = EVALUABLE_PARQUET.with_name(EVALUABLE_PARQUET.name + ".sha256")

PRIOR_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4i0_nested_scv_37_record_exploratory_materialization_v1/"
    "stage6c_4i0_nested_scv_37_record_exploratory_manifest_v1.json"
)
PRIOR_MANIFEST_SHA256 = "651b2fd86cfb24a6a07745cd1737821e77d1b858182afd7863a90b121b5b25ae"

EXPECTED_HASHES = {
    "stage4b_weak_label_table": "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8",
    "stage4b_manifest": "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f",
    "stage4c_full_model": "0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30",
    "stage4c_no_star_model": "6c3fe4fc7fe8fdde7b8f0f0d608c48e66a07945effb8c67c6b98d35e1955257c",
    "stage4c_manifest": "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee",
    "stage6b_evaluable": "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038",
}

EXPECTED_STAGE4B_ROWS = 71_659
EXPECTED_STAGE4B_COLUMNS = 43
EXPECTED_STAGE6B_ROWS = 66_636
EXPECTED_STAGE6B_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

EXPECTED_WEAK_LABEL_ACCOUNTING = {
    "full": {"stable": 61_842, "unstable": 5_723, "unlabeled": 4_094, "eligible": 67_565},
    "no_star": {"stable": 61_298, "unstable": 1_850, "unlabeled": 8_511, "eligible": 63_148},
}

EXPECTED_HELD_OUT_ACCOUNTING = {
    "BRCA1": {"rows": 21_594, "events": 2_023, "negatives": 19_571},
    "BRCA2": {"rows": 34_152, "events": 3_960, "negatives": 30_192},
    "MLH1": {"rows": 8_701, "events": 425, "negatives": 8_276},
}

EXPECTED_TRAINING_ACCOUNTING = {
    ("BRCA1", "full_ges"): {"rows": 41_482, "stable": 39_985, "unstable": 1_497, "iterations": 24},
    ("BRCA1", "no_star_ges"): {"rows": 40_661, "stable": 39_436, "unstable": 1_225, "iterations": 20},
    ("BRCA2", "full_ges"): {"rows": 32_490, "stable": 28_084, "unstable": 4_406, "iterations": 24},
    ("BRCA2", "no_star_ges"): {"rows": 28_319, "stable": 27_536, "unstable": 783, "iterations": 21},
    ("MLH1", "full_ges"): {"rows": 56_738, "stable": 51_245, "unstable": 5_493, "iterations": 22},
    ("MLH1", "no_star_ges"): {"rows": 52_936, "stable": 51_246, "unstable": 1_690, "iterations": 19},
}

PRIMARY_GENES = ["BRCA1", "BRCA2", "MLH1"]
EXPLORATORY_GENE = "EGFR"
OUTCOME_COLUMN = "primary_future_instability"
GENE_COLUMN = "target_gene"
KEY_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]
NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

MODEL_SPECS = OrderedDict([
    ("logo_full_ges", {"display_name": "LOGO Full GES", "score_column": "logo_full_ges_instability_risk"}),
    ("logo_no_star_ges", {"display_name": "LOGO No-star GES", "score_column": "logo_no_star_ges_instability_risk"}),
    ("review_stars", {"display_name": "Review stars", "score_column": "review_stars_instability_risk"}),
    ("combined_metadata", {"display_name": "Combined metadata", "score_column": "combined_metadata_instability_risk"}),
])
PAIRED_COMPARATORS = ["logo_no_star_ges", "review_stars", "combined_metadata"]

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
BOOTSTRAP_PROGRESS_INTERVAL = 50
CI_QUANTILES = (0.025, 0.975)
MINIMUM_VALID_REPLICATES = 1_000

TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4j0_leave_one_gene_out_validation_materialization_v1"
)
QC_DIR = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4j0_leave_one_gene_out_validation_materialization_v1"
)
MANIFEST_DIR = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4j0_leave_one_gene_out_validation_materialization_v1"
)
for directory in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

P = {
    "weak_label_accounting": TABLE_DIR / "stage6c_logo_weak_label_accounting_v1.csv",
    "training_accounting": TABLE_DIR / "stage6c_logo_training_accounting_v1.csv",
    "held_out_accounting": TABLE_DIR / "stage6c_logo_held_out_test_accounting_v1.csv",
    "held_out_predictions": TABLE_DIR / "stage6c_logo_held_out_predictions_v1.parquet",
    "score_summary": TABLE_DIR / "stage6c_logo_score_point_estimate_summary_v1.csv",
    "bootstrap_replicates": TABLE_DIR / "stage6c_logo_bootstrap_replicates_v1.parquet",
    "model_intervals": TABLE_DIR / "stage6c_logo_model_bootstrap_intervals_v1.csv",
    "paired_inference": TABLE_DIR / "stage6c_logo_paired_inference_holm_v1.csv",
    "historical_results": TABLE_DIR / "stage6c_logo_historical_recorded_results_v1.csv",
    "concordance": TABLE_DIR / "stage6c_logo_historical_vs_reproduced_concordance_v1.csv",
    "limitations": TABLE_DIR / "stage6c_logo_interpretation_boundaries_v1.csv",
    "qc": QC_DIR / "stage6c_4j0_leave_one_gene_out_validation_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_4j0_leave_one_gene_out_validation_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text(encoding="utf-8"))["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text(encoding="utf-8"))["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def native(value):
    if isinstance(value, dict):
        return {str(k): native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return native(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    new_hash = sha(temporary)
    if path.exists():
        if sha(path) != new_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical artifact: {path}")
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)
    return sha(path)


def write_csv(path: Path, frame: pd.DataFrame) -> str:
    payload = frame.to_csv(index=False, lineterminator="\n", float_format="%.12g").encode("utf-8")
    return stable_write_bytes(path, payload)


def write_json(path: Path, obj) -> str:
    payload = (
        json.dumps(native(obj), indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    frame.to_parquet(temporary, index=False, compression="zstd", engine="pyarrow")
    if path.exists():
        existing = pd.read_parquet(path)
        fresh = pd.read_parquet(temporary)
        pd.testing.assert_frame_equal(existing, fresh, check_dtype=True, check_exact=True, check_like=False)
        temporary.unlink()
    else:
        os.replace(temporary, path)
    return sha(path)


def sidecar(path: Path) -> Path:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    stable_write_bytes(output, f"{sha(path)}  {path.name}\n".encode("utf-8"))
    return output


def sidecar_hash(path: Path) -> str:
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", Path(path).read_text(encoding="utf-8"))
    if not matches:
        raise RuntimeError(f"No SHA-256 found in sidecar: {path}")
    return matches[0].lower()


def sidecar_ok(path: Path) -> bool:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    return output.exists() and sidecar_hash(output) == sha(path)


def normalize_gene(value) -> str:
    text = str(value).strip().upper()
    return text if text in {"BRCA1", "BRCA2", "MLH1", "EGFR"} else ""


def normalize_rcv(series: pd.Series) -> pd.Series:
    return series.astype("string").str.upper().str.strip().str.extract(r"(RCV\d+)", expand=False)


def gene_from_json(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return ""
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = [part.strip() for part in re.split(r"[,;|]", text) if part.strip()]
    if isinstance(parsed, dict):
        candidates = list(parsed.keys()) + list(parsed.values())
    elif isinstance(parsed, (list, tuple, set, np.ndarray, pd.Series)):
        candidates = list(parsed)
    else:
        candidates = [parsed]
    genes = []
    for candidate in candidates:
        if isinstance(candidate, (list, tuple, set, dict)):
            nested = gene_from_json(candidate)
            if nested:
                genes.append(nested)
            continue
        gene = normalize_gene(candidate)
        if gene:
            genes.append(gene)
    genes = sorted(set(genes))
    return genes[0] if len(genes) == 1 else ""


def extract_sklearn_pipeline(artifact, artifact_name: str):
    if isinstance(artifact, Pipeline):
        return artifact, "<top-level>"
    preferred_keys = (
        "pipeline", "model_pipeline", "fitted_pipeline", "sklearn_pipeline",
        "model", "estimator", "classifier",
    )
    if isinstance(artifact, dict):
        for key in preferred_keys:
            if key in artifact and isinstance(artifact[key], Pipeline):
                return artifact[key], f"[{key!r}]"
    matches = []
    visited = set()
    def walk(obj, location: str, depth: int = 0):
        if depth > 8 or id(obj) in visited:
            return
        visited.add(id(obj))
        if isinstance(obj, Pipeline):
            matches.append((location, obj))
            return
        if isinstance(obj, dict):
            keys = [key for key in preferred_keys if key in obj]
            keys += sorted([key for key in obj if key not in keys], key=lambda x: str(x))
            for key in keys:
                walk(obj[key], f"{location}[{key!r}]", depth + 1)
            return
        if isinstance(obj, (list, tuple)):
            for index, item in enumerate(obj):
                walk(item, f"{location}[{index}]", depth + 1)
            return
        for attr in preferred_keys:
            if hasattr(obj, attr):
                try:
                    walk(getattr(obj, attr), f"{location}.{attr}", depth + 1)
                except Exception:
                    pass
    walk(artifact, "<top-level>")
    unique = {}
    for location, pipeline in matches:
        unique.setdefault(id(pipeline), (location, pipeline))
    unique_matches = list(unique.values())
    if len(unique_matches) == 1:
        return unique_matches[0][1], unique_matches[0][0]
    if not unique_matches:
        raise TypeError(f"No sklearn Pipeline found in frozen {artifact_name} artifact.")
    raise TypeError(
        f"Multiple distinct sklearn Pipelines found in frozen {artifact_name} artifact: "
        f"{[location for location, _ in unique_matches]}"
    )


def find_pipeline_component(pipeline: Pipeline, expected_type):
    matches = [step for _, step in pipeline.steps if isinstance(step, expected_type)]
    if len(matches) != 1:
        raise AssertionError(
            f"Expected exactly one {expected_type.__name__} in pipeline; found {len(matches)}."
        )
    return matches[0]


def stable_class_probability(model: Pipeline, features: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(features)
    classifier = find_pipeline_component(model, LogisticRegression)
    classes = np.asarray(classifier.classes_)
    stable_positions = np.flatnonzero(classes == 1)
    if len(stable_positions) != 1:
        raise AssertionError(f"Model classes do not contain exactly one stable class 1: {classes}")
    return probabilities[:, int(stable_positions[0])]


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.quantile(values, CI_QUANTILES)
    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan
    n = len(differences)
    lower_tail = (np.count_nonzero(differences <= 0.0) + 1) / (n + 1)
    upper_tail = (np.count_nonzero(differences >= 0.0) + 1) / (n + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid_positions = np.where(np.isfinite(p_values))[0]
    if len(valid_positions) == 0:
        return adjusted
    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    m = len(valid_p)
    running_max = 0.0
    for rank, position_within_valid in enumerate(order):
        original_position = valid_positions[position_within_valid]
        raw_adjusted = (m - rank) * valid_p[position_within_valid]
        running_max = max(running_max, raw_adjusted)
        adjusted[original_position] = min(1.0, running_max)
    return adjusted


def interval_status(lower: float, upper: float, positive_label: str, negative_label: str) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive_label
    if upper < 0.0:
        return negative_label
    return "interval_includes_null"


def check_close(observed: float, expected: float, digits: int = 6) -> bool:
    tolerance = 0.5 * (10 ** (-digits)) + 1e-12
    return bool(abs(float(observed) - float(expected)) <= tolerance)


# --------------------------------------------------------------------------------------------------
# 3. EXACT TIE-AWARE METRIC CACHE USED FOR PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

def construct_score_group_cache(scores: np.ndarray, outcomes: np.ndarray) -> dict:
    scores = np.asarray(scores, dtype=np.float64)
    outcomes = np.asarray(outcomes, dtype=np.int8)
    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)
    total_group_matrix = csr_matrix(
        (np.ones(n_rows, dtype=np.float64), (group_index, row_positions)),
        shape=(n_groups, n_rows),
    )
    positive_positions = np.flatnonzero(outcomes == 1)
    positive_group_matrix = csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (group_index[positive_positions], positive_positions),
        ),
        shape=(n_groups, n_rows),
    )
    return {
        "unique_scores": unique_scores,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)
    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals
    group_total_counts = np.asarray(cache["total_group_matrix"] @ count_matrix.T, dtype=np.float64)
    group_positive_counts = np.asarray(cache["positive_group_matrix"] @ count_matrix.T, dtype=np.float64)
    group_negative_counts = group_total_counts - group_positive_counts
    valid = (positive_totals > 0.0) & (negative_totals > 0.0)
    cumulative_negatives_before = np.cumsum(group_negative_counts, axis=0) - group_negative_counts
    concordant_numerator = np.sum(
        group_positive_counts * (cumulative_negatives_before + 0.5 * group_negative_counts),
        axis=0,
    )
    auc_denominator = positive_totals * negative_totals
    auroc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(concordant_numerator, auc_denominator, out=auroc, where=valid)
    positive_desc = group_positive_counts[::-1, :]
    total_desc = group_total_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)
    precision = np.zeros_like(cumulative_positive, dtype=np.float64)
    np.divide(cumulative_positive, cumulative_total, out=precision, where=cumulative_total > 0.0)
    ap_numerator = np.sum(precision * positive_desc, axis=0)
    auprc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(ap_numerator, positive_totals, out=auprc, where=valid)
    return auprc, auroc


# --------------------------------------------------------------------------------------------------
# 4. CRYPTOGRAPHIC PREFLIGHT BEFORE ANY TEMPORAL OUTCOME IS LOADED
# --------------------------------------------------------------------------------------------------

required_files = [
    WEAK_LABEL_TABLE,
    STAGE4B_MANIFEST,
    FULL_MODEL_PATH,
    NO_STAR_MODEL_PATH,
    STAGE4C_MANIFEST,
    EVALUABLE_PARQUET,
    EVALUABLE_SIDECAR,
    PRIOR_MANIFEST,
    PRIOR_MANIFEST.with_name(PRIOR_MANIFEST.name + ".sha256"),
]
for path in required_files:
    if not path.exists():
        raise FileNotFoundError(path)

source_paths = OrderedDict([
    ("stage4b_weak_label_table", WEAK_LABEL_TABLE),
    ("stage4b_manifest", STAGE4B_MANIFEST),
    ("stage4c_full_model", FULL_MODEL_PATH),
    ("stage4c_no_star_model", NO_STAR_MODEL_PATH),
    ("stage4c_manifest", STAGE4C_MANIFEST),
    ("stage6b_evaluable", EVALUABLE_PARQUET),
])
observed_hashes = {}
for key, path in source_paths.items():
    observed = sha(path)
    if observed != EXPECTED_HASHES[key]:
        raise RuntimeError(f"SHA-256 mismatch for {key}: {observed}")
    observed_hashes[key] = observed

if sidecar_hash(EVALUABLE_SIDECAR) != observed_hashes["stage6b_evaluable"]:
    raise RuntimeError("Stage 6B evaluable sidecar verification failed.")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA256 or not sidecar_ok(PRIOR_MANIFEST):
    raise RuntimeError("Prior Cell 6C-4I0 manifest verification failed.")
prior_manifest_readback = json.loads(PRIOR_MANIFEST.read_text(encoding="utf-8"))
if prior_manifest_readback.get("decision") != (
    "PASS_STAGE6C_NESTED_SCV_37_RECORD_EXPLORATORY_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
):
    raise RuntimeError("Prior Cell 6C-4I0 decision mismatch.")

stage4b_metadata = pq.ParquetFile(WEAK_LABEL_TABLE).metadata
if (stage4b_metadata.num_rows, stage4b_metadata.num_columns) != (
    EXPECTED_STAGE4B_ROWS,
    EXPECTED_STAGE4B_COLUMNS,
):
    raise RuntimeError(
        f"Unexpected Stage 4B dimensions: {(stage4b_metadata.num_rows, stage4b_metadata.num_columns)}"
    )

stage6b_metadata = pq.ParquetFile(EVALUABLE_PARQUET).metadata
if (stage6b_metadata.num_rows, stage6b_metadata.num_columns) != (
    EXPECTED_STAGE6B_ROWS,
    EXPECTED_STAGE6B_COLUMNS,
):
    raise RuntimeError(
        f"Unexpected Stage 6B dimensions: {(stage6b_metadata.num_rows, stage6b_metadata.num_columns)}"
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD AND VERIFY STAGE 4B; RECOVER EXACT STAGE 4C PIPELINES
# --------------------------------------------------------------------------------------------------

stage4b_columns = pq.ParquetFile(WEAK_LABEL_TABLE).schema_arrow.names
required_stage4b_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    "full_training_eligible",
    "full_weak_label_binary",
    "no_star_training_eligible",
    "no_star_weak_label_binary",
] + sorted(set(FULL_FEATURES + NO_STAR_FEATURES))
missing_stage4b = [column for column in required_stage4b_columns if column not in stage4b_columns]
if missing_stage4b:
    raise KeyError(f"Missing required Stage 4B columns: {missing_stage4b}")

if GENE_COLUMN in stage4b_columns:
    gene_source_column = GENE_COLUMN
elif "target_genes_json" in stage4b_columns:
    gene_source_column = "target_genes_json"
else:
    raise KeyError("Stage 4B has neither target_gene nor target_genes_json.")

load_stage4b_columns = list(dict.fromkeys(required_stage4b_columns + [gene_source_column]))
weak = pd.read_parquet(WEAK_LABEL_TABLE, columns=load_stage4b_columns).copy()
if len(weak) != EXPECTED_STAGE4B_ROWS:
    raise RuntimeError("Loaded Stage 4B row count mismatch.")
weak[KEY_COLUMN] = normalize_rcv(weak[KEY_COLUMN])
weak[ROW_ORDER_COLUMN] = pd.to_numeric(weak[ROW_ORDER_COLUMN], errors="raise").astype("int64")
if weak[KEY_COLUMN].isna().any() or weak[KEY_COLUMN].nunique() != EXPECTED_STAGE4B_ROWS:
    raise RuntimeError("Stage 4B RCV key verification failed.")
if weak[ROW_ORDER_COLUMN].nunique() != EXPECTED_STAGE4B_ROWS:
    raise RuntimeError("Stage 4B row-order uniqueness failed.")
if not np.all(np.diff(weak[ROW_ORDER_COLUMN].to_numpy()) > 0):
    raise RuntimeError("Stage 4B row order is not strictly increasing.")

if gene_source_column == GENE_COLUMN:
    weak[GENE_COLUMN] = weak[gene_source_column].map(normalize_gene)
else:
    weak[GENE_COLUMN] = weak[gene_source_column].map(gene_from_json)
if weak[GENE_COLUMN].eq("").any():
    raise RuntimeError("Stage 4B gene reconstruction failed.")
if sorted(weak[GENE_COLUMN].unique().tolist()) != ["BRCA1", "BRCA2", "EGFR", "MLH1"]:
    raise RuntimeError("Unexpected Stage 4B genes.")

for feature in sorted(set(FULL_FEATURES + NO_STAR_FEATURES)):
    weak[feature] = pd.to_numeric(weak[feature], errors="coerce").astype(float)
for eligibility_column in ["full_training_eligible", "no_star_training_eligible"]:
    weak[eligibility_column] = weak[eligibility_column].fillna(False).astype(bool)
for label_column in ["full_weak_label_binary", "no_star_weak_label_binary"]:
    weak[label_column] = pd.to_numeric(weak[label_column], errors="coerce")
    nonmissing = weak[label_column].dropna().unique()
    if not set(nonmissing).issubset({0, 1, 0.0, 1.0}):
        raise RuntimeError(f"Unexpected weak-label values in {label_column}.")

weak_label_accounting_rows = []
for model_key, eligibility_column, label_column in [
    ("full", "full_training_eligible", "full_weak_label_binary"),
    ("no_star", "no_star_training_eligible", "no_star_weak_label_binary"),
]:
    eligible = weak[eligibility_column]
    labels = weak[label_column]
    if labels.loc[eligible].isna().any():
        raise RuntimeError(f"Eligible {model_key} rows contain missing labels.")
    noneligible_labeled = int(labels.loc[~eligible].notna().sum())
    stable = int((labels == 1).sum())
    unstable = int((labels == 0).sum())
    unlabeled = int(labels.isna().sum())
    eligible_count = int(eligible.sum())
    expected = EXPECTED_WEAK_LABEL_ACCOUNTING[model_key]
    if (stable, unstable, unlabeled, eligible_count) != (
        expected["stable"], expected["unstable"], expected["unlabeled"], expected["eligible"]
    ):
        raise RuntimeError(f"{model_key} weak-label accounting mismatch.")
    weak_label_accounting_rows.append({
        "model_pathway": model_key,
        "stable_labels": stable,
        "unstable_labels": unstable,
        "unlabeled_rows": unlabeled,
        "training_eligible_rows": eligible_count,
        "noneligible_rows_with_numeric_label": noneligible_labeled,
    })
weak_label_accounting = pd.DataFrame(weak_label_accounting_rows)

frozen_full_artifact = joblib.load(FULL_MODEL_PATH)
frozen_no_star_artifact = joblib.load(NO_STAR_MODEL_PATH)
frozen_full_pipeline, full_pipeline_location = extract_sklearn_pipeline(frozen_full_artifact, "full")
frozen_no_star_pipeline, no_star_pipeline_location = extract_sklearn_pipeline(
    frozen_no_star_artifact, "no-star"
)

for pipeline_name, pipeline, expected_features in [
    ("full", frozen_full_pipeline, FULL_FEATURES),
    ("no_star", frozen_no_star_pipeline, NO_STAR_FEATURES),
]:
    find_pipeline_component(pipeline, SimpleImputer)
    find_pipeline_component(pipeline, StandardScaler)
    classifier = find_pipeline_component(pipeline, LogisticRegression)
    if getattr(pipeline, "n_features_in_", len(expected_features)) != len(expected_features):
        raise RuntimeError(f"Frozen {pipeline_name} feature count mismatch.")
    if set(np.asarray(classifier.classes_).tolist()) != {0, 1}:
        raise RuntimeError(f"Frozen {pipeline_name} class set mismatch.")

full_classifier = find_pipeline_component(frozen_full_pipeline, LogisticRegression)
no_star_classifier = find_pipeline_component(frozen_no_star_pipeline, LogisticRegression)
frozen_model_settings = {
    "full_pipeline_location": full_pipeline_location,
    "no_star_pipeline_location": no_star_pipeline_location,
    "full_solver": full_classifier.solver,
    "full_penalty": full_classifier.penalty,
    "full_C": float(full_classifier.C),
    "full_max_iter": int(full_classifier.max_iter),
    "full_tol": float(full_classifier.tol),
    "full_class_weight": full_classifier.class_weight,
    "full_random_state": full_classifier.random_state,
    "no_star_solver": no_star_classifier.solver,
    "no_star_penalty": no_star_classifier.penalty,
    "no_star_C": float(no_star_classifier.C),
    "no_star_max_iter": int(no_star_classifier.max_iter),
    "no_star_tol": float(no_star_classifier.tol),
    "no_star_class_weight": no_star_classifier.class_weight,
    "no_star_random_state": no_star_classifier.random_state,
}


# --------------------------------------------------------------------------------------------------
# 6. FIT ALL SIX LOGO MODELS USING T0 WEAK LABELS ONLY
#    IMPORTANT: THE STAGE 6B TEMPORAL OUTCOME HAS NOT BEEN OPENED.
# --------------------------------------------------------------------------------------------------

training_accounting_rows = []
prediction_frames = []
fit_start = time.time()

for held_out_gene in PRIMARY_GENES:
    training_genes = [gene for gene in PRIMARY_GENES if gene != held_out_gene]
    training_gene_text = "+".join(training_genes)
    split_predictions = weak.loc[
        weak[GENE_COLUMN].eq(held_out_gene),
        [KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN] + sorted(set(FULL_FEATURES + NO_STAR_FEATURES)),
    ].copy()
    if split_predictions.empty:
        raise RuntimeError(f"No rows found for held-out gene {held_out_gene}.")

    for pathway, template_pipeline, features, eligibility_column, label_column, output_column in [
        (
            "full_ges", frozen_full_pipeline, FULL_FEATURES,
            "full_training_eligible", "full_weak_label_binary",
            "logo_full_ges_instability_risk",
        ),
        (
            "no_star_ges", frozen_no_star_pipeline, NO_STAR_FEATURES,
            "no_star_training_eligible", "no_star_weak_label_binary",
            "logo_no_star_ges_instability_risk",
        ),
    ]:
        train_mask = weak[GENE_COLUMN].isin(training_genes) & weak[eligibility_column]
        train = weak.loc[train_mask].copy()
        y_train = train[label_column].astype(int).to_numpy()
        if len(np.unique(y_train)) != 2:
            raise RuntimeError(f"{held_out_gene}/{pathway} training labels lack both classes.")
        stable_labels = int((y_train == 1).sum())
        unstable_labels = int((y_train == 0).sum())
        model = clone(template_pipeline)
        model.fit(train[features], y_train)
        classifier = find_pipeline_component(model, LogisticRegression)
        n_iter = int(np.max(np.asarray(classifier.n_iter_)))
        converged = n_iter < int(classifier.max_iter)
        if not converged:
            raise RuntimeError(f"{held_out_gene}/{pathway} did not converge.")
        p_stable = stable_class_probability(model, split_predictions[features])
        risk = 1.0 - p_stable
        if not np.isfinite(risk).all() or ((risk < 0.0) | (risk > 1.0)).any():
            raise RuntimeError(f"Invalid LOGO risk for {held_out_gene}/{pathway}.")
        split_predictions[output_column] = risk
        expected_training = EXPECTED_TRAINING_ACCOUNTING[(held_out_gene, pathway)]
        if (
            len(train), stable_labels, unstable_labels, n_iter
        ) != (
            expected_training["rows"], expected_training["stable"],
            expected_training["unstable"], expected_training["iterations"]
        ):
            raise RuntimeError(
                f"Historical training-accounting mismatch for {held_out_gene}/{pathway}: "
                f"{(len(train), stable_labels, unstable_labels, n_iter)}"
            )
        training_accounting_rows.append({
            "held_out_gene": held_out_gene,
            "training_genes": training_gene_text,
            "model_pathway": pathway,
            "training_rows": int(len(train)),
            "stable_weak_labels": stable_labels,
            "unstable_weak_labels": unstable_labels,
            "training_prevalence_stable": stable_labels / len(train),
            "features": ";".join(features),
            "solver": classifier.solver,
            "penalty": classifier.penalty,
            "C": float(classifier.C),
            "max_iter": int(classifier.max_iter),
            "iterations_used": n_iter,
            "converged": converged,
            "t1_outcome_loaded_during_fit": False,
        })
        del train, y_train, model

    prediction_frames.append(
        split_predictions[
            [
                KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN,
                "logo_full_ges_instability_risk",
                "logo_no_star_ges_instability_risk",
            ]
        ].copy()
    )

logo_predictions_all_t0 = pd.concat(prediction_frames, ignore_index=True)
training_accounting = pd.DataFrame(training_accounting_rows)
if logo_predictions_all_t0[KEY_COLUMN].duplicated().any():
    raise RuntimeError("LOGO prediction table contains duplicate RCV keys.")
if sorted(logo_predictions_all_t0[GENE_COLUMN].unique().tolist()) != PRIMARY_GENES:
    raise RuntimeError("LOGO prediction table does not contain exactly the primary genes.")
fit_elapsed = time.time() - fit_start


# --------------------------------------------------------------------------------------------------
# 7. ONLY NOW LOAD THE FROZEN STAGE 6B TEMPORAL OUTCOME AND COMPARATORS
# --------------------------------------------------------------------------------------------------

required_stage6b_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    GENE_COLUMN,
    OUTCOME_COLUMN,
    "review_stars_instability_risk",
    "combined_metadata_instability_risk",
]
stage6b_columns = pq.ParquetFile(EVALUABLE_PARQUET).schema_arrow.names
missing_stage6b = [column for column in required_stage6b_columns if column not in stage6b_columns]
if missing_stage6b:
    raise KeyError(f"Missing required Stage 6B columns: {missing_stage6b}")

outcome = pd.read_parquet(EVALUABLE_PARQUET, columns=required_stage6b_columns).copy()
outcome[KEY_COLUMN] = normalize_rcv(outcome[KEY_COLUMN])
outcome[ROW_ORDER_COLUMN] = pd.to_numeric(outcome[ROW_ORDER_COLUMN], errors="raise").astype("int64")
outcome[GENE_COLUMN] = outcome[GENE_COLUMN].map(normalize_gene)
outcome[OUTCOME_COLUMN] = pd.to_numeric(outcome[OUTCOME_COLUMN], errors="raise").astype(int)
if len(outcome) != EXPECTED_STAGE6B_ROWS or outcome[KEY_COLUMN].nunique() != EXPECTED_STAGE6B_ROWS:
    raise RuntimeError("Stage 6B loaded cohort verification failed.")
if set(outcome[OUTCOME_COLUMN].unique()) - {0, 1}:
    raise RuntimeError("Stage 6B outcome is not binary.")
observed_events = int(outcome[OUTCOME_COLUMN].sum())
observed_negatives = int((outcome[OUTCOME_COLUMN] == 0).sum())
if (observed_events, observed_negatives) != (EXPECTED_EVENTS, EXPECTED_NEGATIVES):
    raise RuntimeError("Stage 6B event accounting mismatch.")
for column in ["review_stars_instability_risk", "combined_metadata_instability_risk"]:
    outcome[column] = pd.to_numeric(outcome[column], errors="raise").astype(float)
    values = outcome[column].to_numpy()
    if not np.isfinite(values).all() or ((values < 0.0) | (values > 1.0)).any():
        raise RuntimeError(f"Invalid Stage 6B comparator: {column}")

primary_outcome = outcome.loc[outcome[GENE_COLUMN].isin(PRIMARY_GENES)].copy()
analysis = primary_outcome.merge(
    logo_predictions_all_t0,
    on=[KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN],
    how="left",
    validate="one_to_one",
    indicator=True,
)
if not analysis["_merge"].eq("both").all():
    raise RuntimeError("At least one primary-gene outcome row lacks LOGO predictions.")
analysis = analysis.drop(columns="_merge").sort_values(ROW_ORDER_COLUMN, kind="mergesort").reset_index(drop=True)

held_out_accounting_rows = []
for gene in PRIMARY_GENES:
    gene_df = analysis.loc[analysis[GENE_COLUMN].eq(gene)]
    rows = int(len(gene_df))
    events = int(gene_df[OUTCOME_COLUMN].sum())
    negatives = rows - events
    expected = EXPECTED_HELD_OUT_ACCOUNTING[gene]
    if (rows, events, negatives) != (expected["rows"], expected["events"], expected["negatives"]):
        raise RuntimeError(f"Held-out accounting mismatch for {gene}.")
    held_out_accounting_rows.append({
        "held_out_gene": gene,
        "rows": rows,
        "events": events,
        "negatives": negatives,
        "event_prevalence": events / rows,
        "bootstrap_attempts": N_BOOTSTRAP,
    })
held_out_accounting = pd.DataFrame(held_out_accounting_rows)

held_out_predictions = analysis[
    [
        KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN, OUTCOME_COLUMN,
        "logo_full_ges_instability_risk", "logo_no_star_ges_instability_risk",
        "review_stars_instability_risk", "combined_metadata_instability_risk",
    ]
].copy()
held_out_predictions = held_out_predictions.rename(columns={GENE_COLUMN: "held_out_gene"})


# --------------------------------------------------------------------------------------------------
# 8. EXACT HELD-OUT POINT ESTIMATES AND 2,000-ATTEMPT PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)
model_interval_rows = []
paired_difference_rows = []
score_summary_rows = []
bootstrap_frames = []
bootstrap_start = time.time()

for held_out_gene in PRIMARY_GENES:
    gene_start = time.time()
    test = analysis.loc[analysis[GENE_COLUMN].eq(held_out_gene)].reset_index(drop=True)
    y = test[OUTCOME_COLUMN].to_numpy(dtype=np.int8)
    n_rows = int(len(test))
    n_events = int(y.sum())
    n_negatives = n_rows - n_events
    prevalence = n_events / n_rows
    training_genes = "+".join([gene for gene in PRIMARY_GENES if gene != held_out_gene])
    if len(np.unique(y)) != 2:
        raise RuntimeError(f"Held-out gene {held_out_gene} lacks both outcome classes.")

    scores = {
        key: test[spec["score_column"]].to_numpy(dtype=np.float64)
        for key, spec in MODEL_SPECS.items()
    }
    caches = {key: construct_score_group_cache(score, y) for key, score in scores.items()}
    original_counts = np.ones((1, n_rows), dtype=np.int16)
    original_positive_total = np.array([n_events], dtype=np.float64)
    point_metrics = {}

    print(
        f"\nPreparing held-out {held_out_gene}: {n_rows:,} rows, "
        f"{n_events:,} events, {n_negatives:,} negatives"
    )
    for model_key, spec in MODEL_SPECS.items():
        score = scores[model_key]
        sklearn_auprc = float(average_precision_score(y, score))
        sklearn_auroc = float(roc_auc_score(y, score))
        fast_auprc, fast_auroc = calculate_grouped_weighted_metrics(
            caches[model_key], original_counts, original_positive_total
        )
        if not np.isclose(fast_auprc[0], sklearn_auprc, rtol=1e-11, atol=1e-12):
            raise RuntimeError(f"Fast AUPRC validation failed for {held_out_gene}/{model_key}.")
        if not np.isclose(fast_auroc[0], sklearn_auroc, rtol=1e-11, atol=1e-12):
            raise RuntimeError(f"Fast AUROC validation failed for {held_out_gene}/{model_key}.")
        point_metrics[model_key] = {"auprc": sklearn_auprc, "auroc": sklearn_auroc}
        score_summary_rows.append({
            "held_out_gene": held_out_gene,
            "training_genes": training_genes,
            "model_key": model_key,
            "model": spec["display_name"],
            "unique_score_values": int(len(np.unique(score))),
            "score_min": float(np.min(score)),
            "score_max": float(np.max(score)),
            "score_mean": float(np.mean(score)),
            "mean_score_events": float(np.mean(score[y == 1])),
            "mean_score_negatives": float(np.mean(score[y == 0])),
            "point_auprc": sklearn_auprc,
            "point_auroc": sklearn_auroc,
            "fast_metric_validation": "PASS",
        })
    print("  Exact tie-aware metric validation against scikit-learn: PASS")

    bootstrap_metrics = {
        model_key: {
            "auprc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
            "auroc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
        }
        for model_key in MODEL_SPECS
    }
    bootstrap_prevalence = np.full(N_BOOTSTRAP, np.nan, dtype=np.float64)
    sampled_events = np.full(N_BOOTSTRAP, -1, dtype=np.int32)
    probabilities = np.full(n_rows, 1.0 / n_rows, dtype=np.float64)
    probabilities[-1] = 1.0 - probabilities[:-1].sum()

    for batch_start in range(0, N_BOOTSTRAP, BOOTSTRAP_BATCH_SIZE):
        batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOTSTRAP)
        batch_size = batch_end - batch_start
        counts = rng.multinomial(n_rows, probabilities, size=batch_size)
        if not np.all(counts.sum(axis=1) == n_rows):
            raise RuntimeError(f"Bootstrap sample-size preservation failed for {held_out_gene}.")
        positive_totals = (counts @ y).astype(np.float64)
        sampled_events[batch_start:batch_end] = positive_totals.astype(np.int32)
        valid = (positive_totals > 0.0) & (positive_totals < n_rows)
        bootstrap_prevalence[batch_start:batch_end] = np.where(
            valid, positive_totals / n_rows, np.nan
        )
        for model_key in MODEL_SPECS:
            auprc_values, auroc_values = calculate_grouped_weighted_metrics(
                caches[model_key], counts, positive_totals
            )
            bootstrap_metrics[model_key]["auprc"][batch_start:batch_end] = auprc_values
            bootstrap_metrics[model_key]["auroc"][batch_start:batch_end] = auroc_values
        if batch_end % BOOTSTRAP_PROGRESS_INTERVAL == 0 or batch_end == N_BOOTSTRAP:
            valid_so_far = int(
                np.isfinite(bootstrap_metrics["logo_full_ges"]["auprc"][:batch_end]).sum()
            )
            print(
                f"  Completed {batch_end:,}/{N_BOOTSTRAP:,} replicates | "
                f"valid {valid_so_far:,} | elapsed {time.time() - gene_start:.1f}s"
            )
        del counts, positive_totals

    valid_mask = np.isfinite(bootstrap_metrics["logo_full_ges"]["auprc"])
    valid_replicates = int(valid_mask.sum())
    invalid_replicates = N_BOOTSTRAP - valid_replicates
    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise RuntimeError(f"Only {valid_replicates} valid replicates for {held_out_gene}.")
    for model_key in MODEL_SPECS:
        for metric in ["auprc", "auroc"]:
            if not np.array_equal(np.isfinite(bootstrap_metrics[model_key][metric]), valid_mask):
                raise RuntimeError(f"Paired-validity mismatch for {held_out_gene}/{model_key}/{metric}.")

    replicate_payload = {
        "held_out_gene": np.repeat(held_out_gene, N_BOOTSTRAP),
        "training_genes": np.repeat(training_genes, N_BOOTSTRAP),
        "replicate": np.arange(1, N_BOOTSTRAP + 1, dtype=np.int32),
        "sampled_rows": np.repeat(n_rows, N_BOOTSTRAP).astype(np.int32),
        "sampled_events": sampled_events,
        "sampled_negatives": (n_rows - sampled_events).astype(np.int32),
        "sampled_prevalence": bootstrap_prevalence,
        "valid_two_class_replicate": valid_mask,
    }
    for model_key in MODEL_SPECS:
        replicate_payload[f"{model_key}_auprc"] = bootstrap_metrics[model_key]["auprc"]
        replicate_payload[f"{model_key}_auroc"] = bootstrap_metrics[model_key]["auroc"]
    bootstrap_frames.append(pd.DataFrame(replicate_payload))

    for model_key, spec in MODEL_SPECS.items():
        auprc_values = bootstrap_metrics[model_key]["auprc"]
        auroc_values = bootstrap_metrics[model_key]["auroc"]
        auprc_low, auprc_high = percentile_interval(auprc_values)
        auroc_low, auroc_high = percentile_interval(auroc_values)
        auprc_null_low, auprc_null_high = percentile_interval(auprc_values - bootstrap_prevalence)
        auroc_null_low, auroc_null_high = percentile_interval(auroc_values - 0.50)
        model_interval_rows.append({
            "held_out_gene": held_out_gene,
            "training_genes": training_genes,
            "model_key": model_key,
            "model": spec["display_name"],
            "rows": n_rows,
            "events": n_events,
            "negatives": n_negatives,
            "held_out_prevalence": prevalence,
            "point_auprc": point_metrics[model_key]["auprc"],
            "auprc_ci_lower": auprc_low,
            "auprc_ci_upper": auprc_high,
            "point_auprc_minus_prevalence": point_metrics[model_key]["auprc"] - prevalence,
            "auprc_minus_prevalence_ci_lower": auprc_null_low,
            "auprc_minus_prevalence_ci_upper": auprc_null_high,
            "auprc_null_status": interval_status(
                auprc_null_low, auprc_null_high,
                "supported_above_held_out_prevalence",
                "supported_below_held_out_prevalence",
            ),
            "point_auroc": point_metrics[model_key]["auroc"],
            "auroc_ci_lower": auroc_low,
            "auroc_ci_upper": auroc_high,
            "point_auroc_minus_0_50": point_metrics[model_key]["auroc"] - 0.50,
            "auroc_minus_0_50_ci_lower": auroc_null_low,
            "auroc_minus_0_50_ci_upper": auroc_null_high,
            "auroc_null_status": interval_status(
                auroc_null_low, auroc_null_high,
                "supported_above_0_50",
                "supported_below_0_50",
            ),
            "attempted_bootstrap_replicates": N_BOOTSTRAP,
            "valid_bootstrap_replicates": valid_replicates,
            "invalid_one_class_replicates": invalid_replicates,
        })

    for comparator_key in PAIRED_COMPARATORS:
        comparator_name = MODEL_SPECS[comparator_key]["display_name"]
        for metric, metric_label in [("auprc", "AUPRC"), ("auroc", "AUROC")]:
            differences = (
                bootstrap_metrics["logo_full_ges"][metric]
                - bootstrap_metrics[comparator_key][metric]
            )
            lower, upper = percentile_interval(differences)
            paired_difference_rows.append({
                "metric": metric_label,
                "held_out_gene": held_out_gene,
                "training_genes": training_genes,
                "comparator_key": comparator_key,
                "comparator": comparator_name,
                "comparison": f"LOGO Full GES minus {comparator_name}",
                "rows": n_rows,
                "events": n_events,
                "negatives": n_negatives,
                "point_difference": (
                    point_metrics["logo_full_ges"][metric]
                    - point_metrics[comparator_key][metric]
                ),
                "difference_ci_lower": lower,
                "difference_ci_upper": upper,
                "paired_interval_status": interval_status(
                    lower, upper,
                    "logo_full_ges_supported_higher",
                    "logo_full_ges_supported_lower",
                ),
                "bootstrap_probability_logo_full_greater": float(
                    np.mean(differences[valid_mask] > 0.0)
                ),
                "bootstrap_sign_p_value": bootstrap_sign_pvalue(differences),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_bootstrap_replicates": valid_replicates,
                "invalid_one_class_replicates": invalid_replicates,
            })

    print(
        f"  Held-out {held_out_gene} complete: {valid_replicates:,} valid, "
        f"{invalid_replicates:,} invalid one-class replicates"
    )

bootstrap_elapsed = time.time() - bootstrap_start
bootstrap_replicates = pd.concat(bootstrap_frames, ignore_index=True)
score_summary = pd.DataFrame(score_summary_rows)
model_intervals = pd.DataFrame(model_interval_rows)
paired_inference = pd.DataFrame(paired_difference_rows)

paired_inference["holm_adjusted_bootstrap_sign_p"] = np.nan
for metric in ["AUPRC", "AUROC"]:
    mask = paired_inference["metric"].eq(metric)
    p_values = paired_inference.loc[mask, "bootstrap_sign_p_value"].to_numpy(float)
    if len(p_values) != 9:
        raise RuntimeError(f"{metric} Holm family has {len(p_values)} tests; expected 9.")
    paired_inference.loc[mask, "holm_adjusted_bootstrap_sign_p"] = holm_adjust(p_values)
paired_inference["holm_supported_at_0_05"] = (
    paired_inference["holm_adjusted_bootstrap_sign_p"] < 0.05
)
paired_inference["multiplicity_family"] = "three_primary_held_out_genes_x_three_comparators"
if paired_inference["holm_adjusted_bootstrap_sign_p"].isna().any():
    raise RuntimeError("At least one Holm-adjusted value is missing.")
if len(bootstrap_replicates) != 6_000:
    raise RuntimeError("Bootstrap replicate table must contain exactly 6,000 rows.")


# --------------------------------------------------------------------------------------------------
# 9. PRESERVE HISTORICAL RESULTS AND TEST CONCORDANCE
# --------------------------------------------------------------------------------------------------

historical_model_values = [
    # held_out_gene, model_key, AUPRC, AUPRC low, AUPRC high, AUROC, AUROC low, AUROC high
    ("BRCA1", "logo_full_ges", 0.123902, 0.114896, 0.134545, 0.565496, 0.553389, 0.577697),
    ("BRCA1", "logo_no_star_ges", 0.100185, 0.092045, 0.109428, 0.451796, 0.438332, 0.464630),
    ("BRCA1", "review_stars", 0.120649, 0.113817, 0.127508, 0.563717, 0.555184, 0.572350),
    ("BRCA1", "combined_metadata", 0.123918, 0.115051, 0.134686, 0.560728, 0.548326, 0.573005),
    ("BRCA2", "logo_full_ges", 0.125352, 0.119858, 0.131753, 0.527071, 0.518228, 0.536247),
    ("BRCA2", "logo_no_star_ges", 0.109955, 0.104851, 0.115839, 0.438465, 0.428913, 0.448145),
    ("BRCA2", "review_stars", 0.122505, 0.118764, 0.126367, 0.529271, 0.522881, 0.535966),
    ("BRCA2", "combined_metadata", 0.126634, 0.121273, 0.133097, 0.528040, 0.518847, 0.537235),
    ("MLH1", "logo_full_ges", 0.063827, 0.053749, 0.078788, 0.560606, 0.534366, 0.585758),
    ("MLH1", "logo_no_star_ges", 0.053943, 0.046508, 0.066230, 0.504515, 0.477880, 0.530766),
    ("MLH1", "review_stars", 0.053313, 0.048379, 0.058551, 0.544057, 0.528931, 0.557877),
    ("MLH1", "combined_metadata", 0.068482, 0.056845, 0.086815, 0.560377, 0.534812, 0.586346),
]

historical_rows = []
for gene, model_key, ap, ap_low, ap_high, auc, auc_low, auc_high in historical_model_values:
    for metric_name, field_name, value in [
        ("AUPRC", "point", ap),
        ("AUPRC", "ci_lower", ap_low),
        ("AUPRC", "ci_upper", ap_high),
        ("AUROC", "point", auc),
        ("AUROC", "ci_lower", auc_low),
        ("AUROC", "ci_upper", auc_high),
    ]:
        historical_rows.append({
            "result_family": "model_interval",
            "held_out_gene": gene,
            "model_key": model_key,
            "metric": metric_name,
            "field": field_name,
            "historical_value": value,
            "recorded_digits": 6,
            "source": "Version 6.0/7.0 technical record, Cell 6C-3H0",
        })

historical_paired_values = [
    # gene, comparator_key, metric, point, low, high, Holm p
    ("BRCA1", "logo_no_star_ges", "AUPRC", 0.023717, 0.020060, 0.027722, 0.008996),
    ("BRCA1", "review_stars", "AUPRC", 0.003253, -0.004002, 0.011937, 0.707646),
    ("BRCA1", "combined_metadata", "AUPRC", -0.000016, -0.003140, 0.003370, 0.957521),
    ("BRCA2", "logo_no_star_ges", "AUPRC", 0.015397, 0.014101, 0.016740, 0.008996),
    ("BRCA2", "review_stars", "AUPRC", 0.002847, -0.001113, 0.007479, 0.461769),
    ("BRCA2", "combined_metadata", "AUPRC", -0.001282, -0.002573, 0.000036, 0.215892),
    ("MLH1", "logo_no_star_ges", "AUPRC", 0.009884, 0.006071, 0.014901, 0.008996),
    ("MLH1", "review_stars", "AUPRC", 0.010514, 0.002413, 0.023830, 0.059970),
    ("MLH1", "combined_metadata", "AUPRC", -0.004655, -0.010679, -0.000675, 0.129935),
    ("BRCA1", "logo_no_star_ges", "AUROC", 0.113700, 0.105672, 0.122137, 0.008996),
    ("BRCA1", "review_stars", "AUROC", 0.001779, -0.006651, 0.009688, 1.000000),
    ("BRCA1", "combined_metadata", "AUROC", 0.004768, 0.001525, 0.008014, 0.011994),
    ("BRCA2", "logo_no_star_ges", "AUROC", 0.088605, 0.082611, 0.094385, 0.008996),
    ("BRCA2", "review_stars", "AUROC", -0.002200, -0.008404, 0.003986, 1.000000),
    ("BRCA2", "combined_metadata", "AUROC", -0.000969, -0.003069, 0.001128, 1.000000),
    ("MLH1", "logo_no_star_ges", "AUROC", 0.056091, 0.042249, 0.068811, 0.008996),
    ("MLH1", "review_stars", "AUROC", 0.016549, -0.004204, 0.036021, 0.679660),
    ("MLH1", "combined_metadata", "AUROC", 0.000229, -0.010297, 0.009393, 1.000000),
]
for gene, comparator_key, metric, point, low, high, holm_p in historical_paired_values:
    for field_name, value in [
        ("point_difference", point),
        ("difference_ci_lower", low),
        ("difference_ci_upper", high),
        ("holm_adjusted_bootstrap_sign_p", holm_p),
    ]:
        historical_rows.append({
            "result_family": "paired_inference",
            "held_out_gene": gene,
            "model_key": comparator_key,
            "metric": metric,
            "field": field_name,
            "historical_value": value,
            "recorded_digits": 6,
            "source": "Version 6.0/7.0 technical record, Cell 6C-3H0",
        })
historical_results = pd.DataFrame(historical_rows)

model_lookup = model_intervals.set_index(["held_out_gene", "model_key"])
paired_lookup = paired_inference.set_index(["held_out_gene", "comparator_key", "metric"])
concordance_rows = []
for row in historical_results.itertuples(index=False):
    if row.result_family == "model_interval":
        reproduced_field = {
            ("AUPRC", "point"): "point_auprc",
            ("AUPRC", "ci_lower"): "auprc_ci_lower",
            ("AUPRC", "ci_upper"): "auprc_ci_upper",
            ("AUROC", "point"): "point_auroc",
            ("AUROC", "ci_lower"): "auroc_ci_lower",
            ("AUROC", "ci_upper"): "auroc_ci_upper",
        }[(row.metric, row.field)]
        reproduced = float(model_lookup.loc[(row.held_out_gene, row.model_key), reproduced_field])
    else:
        reproduced = float(
            paired_lookup.loc[(row.held_out_gene, row.model_key, row.metric), row.field]
        )
    reproduced_ok = check_close(reproduced, row.historical_value, row.recorded_digits)
    concordance_rows.append({
        "result_family": row.result_family,
        "held_out_gene": row.held_out_gene,
        "model_or_comparator_key": row.model_key,
        "metric": row.metric,
        "field": row.field,
        "historical_value": row.historical_value,
        "reproduced_value": reproduced,
        "absolute_difference": abs(reproduced - row.historical_value),
        "recorded_digits": row.recorded_digits,
        "value_reproduced_at_recorded_precision": reproduced_ok,
    })
concordance = pd.DataFrame(concordance_rows)
if not concordance["value_reproduced_at_recorded_precision"].all():
    failed = concordance.loc[~concordance["value_reproduced_at_recorded_precision"]]
    raise RuntimeError(
        "Historical LOGO concordance failed:\n" + failed.to_string(index=False)
    )

# Scientific-conclusion checks.
no_star_rows = paired_inference.loc[paired_inference["comparator_key"].eq("logo_no_star_ges")]
full_better_no_star_all = bool(
    (len(no_star_rows) == 6)
    and no_star_rows["paired_interval_status"].eq("logo_full_ges_supported_higher").all()
    and no_star_rows["holm_supported_at_0_05"].all()
)
review_combined_rows = paired_inference.loc[
    paired_inference["comparator_key"].isin(["review_stars", "combined_metadata"])
]
no_consistent_incremental_value = bool(
    not review_combined_rows["holm_supported_at_0_05"].all()
    and (
        review_combined_rows["paired_interval_status"].eq("interval_includes_null").any()
        or review_combined_rows["paired_interval_status"].eq("logo_full_ges_supported_lower").any()
    )
)
only_supported_combined_advantage = paired_inference.loc[
    paired_inference["comparator_key"].eq("combined_metadata")
    & paired_inference["holm_supported_at_0_05"]
]
only_brca1_auroc_supported_vs_combined = bool(
    len(only_supported_combined_advantage) == 1
    and only_supported_combined_advantage.iloc[0]["held_out_gene"] == "BRCA1"
    and only_supported_combined_advantage.iloc[0]["metric"] == "AUROC"
    and only_supported_combined_advantage.iloc[0]["paired_interval_status"]
        == "logo_full_ges_supported_higher"
)

limitations = pd.DataFrame([
    {
        "boundary_key": "weak_absolute_discrimination",
        "statement": (
            "Held-out full-GES discrimination is positive but weak; results support relative ranking "
            "transfer, not strong generalization or clinical utility."
        ),
    },
    {
        "boundary_key": "limited_incremental_value",
        "statement": (
            "Full GES consistently exceeds the no-star ablation but does not consistently exceed review "
            "stars or combined metadata across held-out genes and metrics."
        ),
    },
    {
        "boundary_key": "not_calibrated_probability",
        "statement": "LOGO risk values are not demonstrated calibrated probabilities.",
    },
    {
        "boundary_key": "primary_genes_only",
        "statement": "The primary LOGO design uses BRCA1, BRCA2, and MLH1; EGFR is excluded.",
    },
    {
        "boundary_key": "experiment_2_not_started",
        "statement": "No RAG outcome was accessed and Experiment 2 was not started.",
    },
])


# --------------------------------------------------------------------------------------------------
# 10. QC BEFORE WRITING
# --------------------------------------------------------------------------------------------------

checks = []
def check(name: str, passed: bool, details=None):
    checks.append({"check_name": name, "passed": bool(passed), "details": native(details)})

check("source_hashes", all(observed_hashes[k] == EXPECTED_HASHES[k] for k in EXPECTED_HASHES), observed_hashes)
check("prior_6c4i0_manifest", sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA256, sha(PRIOR_MANIFEST))
check("weak_label_accounting_rows", len(weak_label_accounting) == 2, len(weak_label_accounting))
check("training_accounting_rows", len(training_accounting) == 6, len(training_accounting))
check("training_outcome_blind", not training_accounting["t1_outcome_loaded_during_fit"].any())
check("held_out_accounting_rows", len(held_out_accounting) == 3, len(held_out_accounting))
check("held_out_predictions_rows", len(held_out_predictions) == 64_447, len(held_out_predictions))
check("held_out_predictions_unique_keys", held_out_predictions[KEY_COLUMN].nunique() == len(held_out_predictions))
check("score_summary_rows", len(score_summary) == 12, len(score_summary))
check("model_intervals_rows", len(model_intervals) == 12, len(model_intervals))
check("paired_inference_rows", len(paired_inference) == 18, len(paired_inference))
check("holm_families", paired_inference.groupby("metric").size().to_dict() == {"AUPRC": 9, "AUROC": 9})
check("bootstrap_rows", len(bootstrap_replicates) == 6_000, len(bootstrap_replicates))
check("bootstrap_all_valid", bootstrap_replicates["valid_two_class_replicate"].all())
check("bootstrap_gene_counts", bootstrap_replicates.groupby("held_out_gene").size().to_dict() == {"BRCA1": 2000, "BRCA2": 2000, "MLH1": 2000})
check("historical_concordance", concordance["value_reproduced_at_recorded_precision"].all(), int(concordance["value_reproduced_at_recorded_precision"].sum()))
check("full_better_no_star_all", full_better_no_star_all)
check("no_consistent_incremental_value", no_consistent_incremental_value)
check("only_brca1_auroc_supported_vs_combined", only_brca1_auroc_supported_vs_combined)
check("egfr_excluded", EXPLORATORY_GENE not in set(held_out_predictions["held_out_gene"]))
check("experiment_2_not_started", True)

failed = [item for item in checks if not item["passed"]]
if failed:
    raise RuntimeError("QC failed before writing:\n" + json.dumps(native(failed), indent=2))


# --------------------------------------------------------------------------------------------------
# 11. WRITE, SIDECAR, READ BACK, AND MANIFEST
# --------------------------------------------------------------------------------------------------

write_csv(P["weak_label_accounting"], weak_label_accounting)
write_csv(P["training_accounting"], training_accounting)
write_csv(P["held_out_accounting"], held_out_accounting)
write_parquet(P["held_out_predictions"], held_out_predictions)
write_csv(P["score_summary"], score_summary)
write_parquet(P["bootstrap_replicates"], bootstrap_replicates)
write_csv(P["model_intervals"], model_intervals)
write_csv(P["paired_inference"], paired_inference)
write_csv(P["historical_results"], historical_results)
write_csv(P["concordance"], concordance)
write_csv(P["limitations"], limitations)

readback = {
    "weak_label_accounting": len(pd.read_csv(P["weak_label_accounting"])) == 2,
    "training_accounting": len(pd.read_csv(P["training_accounting"])) == 6,
    "held_out_accounting": len(pd.read_csv(P["held_out_accounting"])) == 3,
    "held_out_predictions": len(pd.read_parquet(P["held_out_predictions"])) == 64_447,
    "score_summary": len(pd.read_csv(P["score_summary"])) == 12,
    "bootstrap_replicates": len(pd.read_parquet(P["bootstrap_replicates"])) == 6_000,
    "model_intervals": len(pd.read_csv(P["model_intervals"])) == 12,
    "paired_inference": len(pd.read_csv(P["paired_inference"])) == 18,
    "historical_results": len(pd.read_csv(P["historical_results"])) == len(historical_results),
    "concordance": len(pd.read_csv(P["concordance"])) == len(concordance),
    "limitations": len(pd.read_csv(P["limitations"])) == 5,
}
if not all(readback.values()):
    raise RuntimeError(f"Table readback failed: {readback}")

qc_payload = {
    "cell_id": "6C-4J0",
    "package_version": "v1",
    "created_utc": CREATED_UTC,
    "analysis": "leave_one_gene_out_validation_materialization",
    "immutable_sources": {
        key: {"path": str(path), "sha256": observed_hashes[key]}
        for key, path in source_paths.items()
    },
    "prior_6c4i0_manifest": {"path": str(PRIOR_MANIFEST), "sha256": sha(PRIOR_MANIFEST)},
    "pipeline_recovery": {
        "full_artifact_type": type(frozen_full_artifact).__name__,
        "full_pipeline_location": full_pipeline_location,
        "no_star_artifact_type": type(frozen_no_star_artifact).__name__,
        "no_star_pipeline_location": no_star_pipeline_location,
        "settings": frozen_model_settings,
    },
    "bootstrap": {
        "method": "paired ordinary row bootstrap with replacement",
        "exact_historical_implementation": (
            "continuous numpy.random.default_rng(42) across BRCA1, BRCA2, MLH1; "
            "rng.multinomial(n_rows, adjusted_uniform_probabilities, size=50)"
        ),
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
        "seed": RANDOM_SEED,
        "attempts_per_held_out_gene": N_BOOTSTRAP,
        "total_attempts": 6_000,
        "valid_replicates": int(bootstrap_replicates["valid_two_class_replicate"].sum()),
        "invalid_one_class_replicates": int((~bootstrap_replicates["valid_two_class_replicate"]).sum()),
        "identical_resamples_across_four_scores_within_gene": True,
        "separate_holm_families": {"AUPRC": 9, "AUROC": 9},
    },
    "checks": checks,
    "table_readback": readback,
    "passed_checks": sum(item["passed"] for item in checks),
    "failed_checks": sum(not item["passed"] for item in checks),
    "decision": (
        "PASS_STAGE6C_LEAVE_ONE_GENE_OUT_VALIDATION_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["qc"], qc_payload)

artifact_keys = [
    "weak_label_accounting", "training_accounting", "held_out_accounting",
    "held_out_predictions", "score_summary", "bootstrap_replicates",
    "model_intervals", "paired_inference", "historical_results", "concordance",
    "limitations", "qc",
]
for key in artifact_keys:
    sidecar(P[key])
    if not sidecar_ok(P[key]):
        raise RuntimeError(f"Sidecar verification failed: {P[key]}")


def artifact_record(key: str) -> dict:
    path = P[key]
    record = {
        "artifact_key": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha(path),
        "bytes": path.stat().st_size,
        "sidecar_path": str(path.with_name(path.name + ".sha256")),
        "sidecar_verified": sidecar_ok(path),
    }
    if path.suffix == ".csv":
        loaded = pd.read_csv(path)
        record.update(rows=len(loaded), columns=loaded.shape[1])
    elif path.suffix == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        record.update(rows=metadata.num_rows, columns=metadata.num_columns)
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True
    return record

manifest = {
    "cell_id": "6C-4J0",
    "package_version": "v1",
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": CREATED_UTC,
    "authorized_category": "leave_one_gene_out_validation_materialization",
    "immutable_sources": qc_payload["immutable_sources"],
    "prior_6c4i0_manifest": qc_payload["prior_6c4i0_manifest"],
    "analysis_lock": {
        "primary_genes": PRIMARY_GENES,
        "exploratory_gene_excluded": EXPLORATORY_GENE,
        "splits": {
            "BRCA1": ["BRCA2", "MLH1"],
            "BRCA2": ["BRCA1", "MLH1"],
            "MLH1": ["BRCA1", "BRCA2"],
        },
        "outcome_loaded_during_fit": False,
        "full_features": FULL_FEATURES,
        "no_star_features": NO_STAR_FEATURES,
        "models": MODEL_SPECS,
        "bootstrap_seed": RANDOM_SEED,
        "bootstrap_attempts_per_gene": N_BOOTSTRAP,
        "bootstrap_batch_size": BOOTSTRAP_BATCH_SIZE,
        "bootstrap_implementation": (
            "one continuous Generator(PCG64) stream; multinomial row-count samples with "
            "adjusted final uniform probability"
        ),
        "paired_comparators": PAIRED_COMPARATORS,
        "holm_families": {"AUPRC": 9, "AUROC": 9},
    },
    "result_summary": {
        "held_out_rows": int(len(held_out_predictions)),
        "bootstrap_total_attempts": 6_000,
        "bootstrap_valid_attempts": int(bootstrap_replicates["valid_two_class_replicate"].sum()),
        "full_better_than_no_star_in_all_six_metric_gene_comparisons": full_better_no_star_all,
        "no_consistent_incremental_value_over_review_or_combined": no_consistent_incremental_value,
        "only_holm_supported_full_advantage_over_combined_metadata": "BRCA1_AUROC",
        "historical_values_reproduced": bool(
            concordance["value_reproduced_at_recorded_precision"].all()
        ),
        "full_logo_point_estimates": {
            gene: {
                "auprc": float(model_lookup.loc[(gene, "logo_full_ges"), "point_auprc"]),
                "auroc": float(model_lookup.loc[(gene, "logo_full_ges"), "point_auroc"]),
            }
            for gene in PRIMARY_GENES
        },
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "artifacts": [artifact_record(key) for key in artifact_keys],
    "scientific_boundary": {
        "t1_outcome_used_during_model_fit": False,
        "egfr_pooled_into_primary_logo": False,
        "frozen_inputs_modified": False,
        "frozen_models_modified": False,
        "feature_definitions_changed": False,
        "scores_recalibrated": False,
        "thresholds_or_weights_changed": False,
        "outcome_definition_changed": False,
        "linkage_decisions_changed": False,
        "row_order_changed": False,
        "cohort_membership_changed": False,
        "experiment_2_started": False,
        "interpretation": (
            "LOGO Full GES retained weak positive held-out ranking in BRCA1, BRCA2, and MLH1 and "
            "consistently exceeded the LOGO no-star ablation. It did not show consistent incremental "
            "value over review stars or combined metadata. These results support weak cross-gene "
            "ranking transfer, not strong generalization, calibration, clinical utility, or RAG safety."
        ),
    },
    "decision": (
        "PASS_STAGE6C_LEAVE_ONE_GENE_OUT_VALIDATION_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "stage6c_materialization_status": "8_of_8_categories_complete",
    "next_authorized_step": "final_integrated_stage6c_package_freeze",
}
manifest_hash = write_json(P["manifest"], manifest)
sidecar(P["manifest"])

manifest_readback = json.loads(P["manifest"].read_text(encoding="utf-8"))
if not sidecar_ok(P["manifest"]):
    raise RuntimeError("Manifest sidecar verification failed.")
if manifest_readback["decision"] != manifest["decision"]:
    raise RuntimeError("Manifest decision readback mismatch.")
if manifest_readback["scientific_boundary"]["experiment_2_started"] is not False:
    raise RuntimeError("Experiment 2 boundary failed.")
for artifact in manifest_readback["artifacts"]:
    path = Path(artifact["path"])
    if sha(path) != artifact["sha256"] or not sidecar_ok(path):
        raise RuntimeError(f"Final artifact verification failed: {path}")

for key, path in source_paths.items():
    if sha(path) != EXPECTED_HASHES[key]:
        raise RuntimeError(f"Immutable source changed during Cell 6C-4J0: {key}")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA256:
    raise RuntimeError("Prior manifest changed during Cell 6C-4J0.")


# --------------------------------------------------------------------------------------------------
# 12. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 170)
print(
    "STAGE 6C STEP 4J — CELL 6C-4J0 — "
    "LEAVE-ONE-GENE-OUT VALIDATION RESULT CATEGORY"
)
print("=" * 170)
for label, key in [
    ("Stage 4B weak-label table", "stage4b_weak_label_table"),
    ("Stage 4B manifest", "stage4b_manifest"),
    ("Stage 4C full model", "stage4c_full_model"),
    ("Stage 4C no-star model", "stage4c_no_star_model"),
    ("Stage 4C manifest", "stage4c_manifest"),
    ("Stage 6B evaluable cohort", "stage6b_evaluable"),
]:
    print(f"{label:40s}: PASS ({observed_hashes[key]})")
print(f"Prior Cell 6C-4I0 manifest              : PASS ({sha(PRIOR_MANIFEST)})")
print(
    "Frozen Stage 4C artifact unwrapping   : "
    f"full={type(frozen_full_artifact).__name__}{full_pipeline_location}; "
    f"no-star={type(frozen_no_star_artifact).__name__}{no_star_pipeline_location}"
)
print("Temporal outcome loaded during fit      : No")
print("Held-out model tuning                   : None")
print("EGFR pooled into primary LOGO           : No")
print(f"Model fitting elapsed                   : {fit_elapsed:.1f}s")
print(f"Bootstrap elapsed                       : {bootstrap_elapsed:.1f}s")
print(f"Bootstrap attempts                      : {N_BOOTSTRAP:,} per held-out gene")
print(f"Valid / invalid total replicates        : {int(bootstrap_replicates.valid_two_class_replicate.sum()):,} / {int((~bootstrap_replicates.valid_two_class_replicate).sum()):,}")
print("Holm families                           : 9 AUPRC + 9 AUROC")
print(f"Historical recorded values              : PASS ({int(concordance.value_reproduced_at_recorded_precision.sum())}/{len(concordance)})")
print(f"Fresh QC                                : PASS ({qc_payload['passed_checks']}/{len(checks)})")

for label, key in [
    ("Weak-label accounting", "weak_label_accounting"),
    ("Training accounting", "training_accounting"),
    ("Held-out test accounting", "held_out_accounting"),
    ("Held-out predictions", "held_out_predictions"),
    ("Score summary", "score_summary"),
    ("Bootstrap replicates", "bootstrap_replicates"),
    ("Model intervals", "model_intervals"),
    ("Paired inference + Holm", "paired_inference"),
    ("Historical results", "historical_results"),
    ("Concordance", "concordance"),
    ("Interpretation boundaries", "limitations"),
    ("QC", "qc"),
    ("Manifest", "manifest"),
]:
    print(f"{label:40s}: {P[key]}")
print(f"Manifest SHA-256                        : {manifest_hash}")

print("\nLOGO TRAINING ACCOUNTING")
print(training_accounting.to_string(index=False))
print("\nHELD-OUT TEST ACCOUNTING")
print(held_out_accounting.to_string(index=False))
print("\nHELD-OUT MODEL-SPECIFIC INTERVALS")
print(
    model_intervals[
        [
            "held_out_gene", "training_genes", "model", "rows", "events", "negatives",
            "held_out_prevalence", "point_auprc", "auprc_ci_lower", "auprc_ci_upper",
            "auprc_null_status", "point_auroc", "auroc_ci_lower", "auroc_ci_upper",
            "auroc_null_status", "valid_bootstrap_replicates", "invalid_one_class_replicates",
        ]
    ].to_string(index=False)
)
print("\nPAIRED LOGO FULL-GES-MINUS-COMPARATOR INFERENCE")
print(
    paired_inference[
        [
            "metric", "held_out_gene", "training_genes", "comparison", "point_difference",
            "difference_ci_lower", "difference_ci_upper", "paired_interval_status",
            "bootstrap_sign_p_value", "holm_adjusted_bootstrap_sign_p",
            "holm_supported_at_0_05", "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ]
    ].to_string(index=False)
)

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print(
    "LOGO Full GES preserved weak positive held-out ranking in BRCA1, BRCA2, and MLH1 and "
    "consistently exceeded the no-star ablation. It did not show consistent incremental value "
    "over review stars or combined metadata. These results support weak cross-gene ranking "
    "transfer, not strong generalization, calibration, clinical utility, or RAG safety."
)

print("\nCELL DECISION")
print(
    "PASS_STAGE6C_LEAVE_ONE_GENE_OUT_VALIDATION_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)
print(
    "All eight Stage 6C result categories are now independently materialized. "
    "The next authorized step is the final integrated Stage 6C package freeze. "
    "Experiment 2 has not started."
)
print("=" * 170)


Mounted at /content/drive

Preparing held-out BRCA1: 21,594 rows, 2,023 events, 19,571 negatives
  Exact tie-aware metric validation against scikit-learn: PASS
  Completed 50/2,000 replicates | valid 50 | elapsed 0.3s
  Completed 100/2,000 replicates | valid 100 | elapsed 0.5s
  Completed 150/2,000 replicates | valid 150 | elapsed 0.7s
  Completed 200/2,000 replicates | valid 200 | elapsed 0.9s
  Completed 250/2,000 replicates | valid 250 | elapsed 1.1s
  Completed 300/2,000 replicates | valid 300 | elapsed 1.2s
  Completed 350/2,000 replicates | valid 350 | elapsed 1.4s
  Completed 400/2,000 replicates | valid 400 | elapsed 1.6s
  Completed 450/2,000 replicates | valid 450 | elapsed 1.8s
  Completed 500/2,000 replicates | valid 500 | elapsed 2.0s
  Completed 550/2,000 replicates | valid 550 | elapsed 2.2s
  Completed 600/2,000 replicates | valid 600 | elapsed 2.5s
  Completed 650/2,000 replicates | valid 650 | elapsed 2.8s
  Completed 700/2,000 replicates | valid 700 | elapsed 3.0s
  